# jscip demo
This cell demonstrates:
- Defining independent and derived parameters
- Building a ParameterBank with constraints
- Default instances
- Sampling instances and parameter vectors
- Converting between ParameterSet, DataFrame, and numpy vectors
- Using log_prob and the new vector-based helpers

In [1]:
import numpy as np
import pandas as pd

from jscip import IndependentParameter, DerivedParameter, ParameterBank, ParameterSet

In [2]:
# Two independent parameters:
# - "alpha": sampled uniformly on [0.0, 2.0]
# - "beta":  sampled uniformly on [-1.0, 1.0]
# One fixed independent parameter:
# - "gamma": fixed at 0.5, not sampled
# One derived parameter:
# - "alpha_plus_beta": alpha + beta

alpha = IndependentParameter(value=1.0, is_sampled=True, range=(0.0, 2.0))
beta = IndependentParameter(value=0.0, is_sampled=True, range=(-1.0, 1.0))
gamma = IndependentParameter(value=0.5, is_sampled=False, range=(0.0, 1.0))

def alpha_plus_beta_fn(ps: ParameterSet) -> float:
    return float(ps["alpha"] + ps["beta"])

alpha_plus_beta = DerivedParameter(function=alpha_plus_beta_fn)

parameters = {
    "alpha": alpha,
    "beta": beta,
    "gamma": gamma,
    "alpha_plus_beta": alpha_plus_beta,
}

# Optional TeX names for plotting / LaTeX labels
texnames = {
    "alpha": r"\alpha",
    "beta": r"\beta",
    "gamma": r"\gamma",
    "alpha_plus_beta": r"\alpha + \beta",
}


In [3]:
# Example constraint: enforce alpha + beta >= 0
def constraint_alpha_plus_beta_nonnegative(ps: ParameterSet) -> bool:
    return float(ps["alpha"] + ps["beta"]) >= 0.0

constraints = [constraint_alpha_plus_beta_nonnegative]

In [4]:
# vector_mode=True means:
# - Parameter vectors only include sampled independent parameters
#   (here: alpha and beta, but not gamma or derived parameters).
bank = ParameterBank(
    parameters=parameters,
    constraints=constraints,
    vector_mode=True,
    texnames=texnames,
)

# Inspect configuration
print("=== ParameterBank summary ===")
print(bank.summary())
print()

print("Names:", bank.names)
print("Sampled (independent) parameters:", bank.sampled)
print("Sampled texnames:", bank.sampled_texnames)
print("Lower bounds:", bank.lower_bounds)
print("Upper bounds:", bank.upper_bounds)
print()


=== ParameterBank summary ===
ParameterBank:
----------------
alpha: IndependentParameter(value=1.0, range=(0.0, 2.0), is_sampled=True)
beta: IndependentParameter(value=0.0, range=(-1.0, 1.0), is_sampled=True)
gamma: IndependentParameter(value=0.5, range=(0.0, 1.0), is_sampled=False)
alpha_plus_beta: DerivedParameter(function=alpha_plus_beta_fn)
Constraints:
----------------
<function constraint_alpha_plus_beta_nonnegative at 0x000002C3FC388180>

Names: ['alpha', 'beta', 'gamma', 'alpha_plus_beta']
Sampled (independent) parameters: ['alpha', 'beta']
Sampled texnames: ['\\alpha', '\\beta']
Lower bounds: [ 0. -1.]
Upper bounds: [2. 1.]



In [5]:
print("=== Default instance (all parameters) ===")
default_instance = bank.get_default_values(return_vector=False)
print(default_instance)
print()

print("=== Default parameter vector (sampled parameters only) ===")
default_vector = bank.get_default_values(return_vector=True)
print("Default vector:", default_vector, "shape:", default_vector.shape)
print()

# The new alias does the same as instance_to_vector, but with clearer naming
default_vector_via_alias = bank.instance_to_vector(default_instance)
print("Default vector via instance_to_vector:", default_vector_via_alias)
print()

# Round-trip: vector -> ParameterSet (using the new alias)
recovered_instance = bank.vector_to_instance(default_vector_via_alias)
print("Recovered instance from vector:")
print(recovered_instance)
print()


=== Default instance (all parameters) ===
ParameterSet(alpha              1.0
beta               0.0
gamma              0.5
alpha_plus_beta    1.0
dtype: float64)

=== Default parameter vector (sampled parameters only) ===
Default vector: [1. 0.] shape: (2,)

Default vector via instance_to_vector: [1. 0.]

Recovered instance from vector:
ParameterSet(alpha              1.0
beta               0.0
gamma              0.5
alpha_plus_beta    1.0
dtype: float64)



In [6]:
print("=== Single sample (ParameterSet) ===")
single_sample = bank.sample(size=None)  # or bank.sample()
print(single_sample)
print()

print("=== Multiple samples as DataFrame (instance mode) ===")
# vector_sampling=True but size is int -> jscip returns a vector array.
# To illustrate DataFrame, temporarily use another bank with vector_sampling=False.
bank_full = ParameterBank(
    parameters=parameters,
    constraints=constraints,
    vector_mode=False,
    texnames=texnames,
)

samples_df = bank_full.sample(size=5)
print(samples_df)
print()

print("=== Multiple samples as parameter vectors (vector / vector mode) ===")
# Back to the original bank with vector_sampling=True
vectors = bank.sample(size=5)  # returns numpy array of shape (5, n_sampled)
print("Sampled vectors shape:", vectors.shape)
print(vectors)
print()


=== Single sample (ParameterSet) ===
[ 1.66070831 -0.89385264]

=== Multiple samples as DataFrame (instance mode) ===
      alpha      beta  gamma  alpha_plus_beta
0  1.199912  0.484549    0.5         1.684461
1  1.782154 -0.064949    0.5         1.717205
2  0.971197 -0.640060    0.5         0.331137
3  1.637901 -0.306084    0.5         1.331817
4  0.099571  0.692909    0.5         0.792480

=== Multiple samples as parameter vectors (vector / vector mode) ===
Sampled vectors shape: (5, 2)
[[0.18732792 0.92611642]
 [0.32143318 0.53739776]
 [0.06115156 0.92326141]
 [0.62286595 0.95974257]
 [1.57006096 0.1834181 ]]



In [7]:
print("=== dataframe_to_vector demo ===")
# Use the DataFrame from bank_full and then restrict to sampled columns of `bank`
# (alpha, beta). This mimics a typical workflow where you have full configs,
# but only some parameters are treated as sampled.
aligned_df = samples_df[bank.sampled]  # ensure correct columns/order
param_matrix = bank.dataframe_to_vector(aligned_df)  # alias for dataframe_to_vector
print("Matrix shape:", param_matrix.shape)
print(param_matrix)
print()

# And convert a single row vector back to a ParameterSet (via vector_to_instance)
vector_example = param_matrix[0]
print("Vector example:", vector_example)
instance_from_vector = bank.vector_to_instance(vector_example)
print("Instance recovered from vector example:")
print(instance_from_vector)
print()



=== dataframe_to_vector demo ===
Matrix shape: (5, 2)
[[ 1.19991208  0.48454884]
 [ 1.78215401 -0.06494863]
 [ 0.97119692 -0.64006023]
 [ 1.63790131 -0.30608395]
 [ 0.09957112  0.69290861]]

Vector example: [1.19991208 0.48454884]
Instance recovered from vector example:
ParameterSet(alpha              1.199912
beta               0.484549
gamma              0.500000
alpha_plus_beta    1.684461
dtype: float64)



In [8]:
print("=== log_prob examples ===")

# 7.1 ParameterSet
lp_default = bank.log_prob(default_instance)
print("log_prob(default_instance):", lp_default)

# 7.2 DataFrame
lp_df = bank.log_prob(aligned_df)
print("log_prob(aligned_df) (array of length =", len(lp_df), "):")
print(lp_df)

# 7.3 numpy arrays in vector mode
#    For vector_sampling=True, 1D arrays must have len == len(bank.sampled).
vector_good = bank.instance_to_vector(instance_from_vector)
lp_vector_good = bank.log_prob(vector_good)
print("log_prob(vector_good):", lp_vector_good)

# 2D array: one row per sample
lp_vectors = bank.log_prob(vectors)
print("log_prob on 2D matrix of vectors, shape", vectors.shape, ":")
print(lp_vectors)


=== log_prob examples ===
log_prob(default_instance): 0.0
log_prob(aligned_df) (array of length = 5 ):
[0. 0. 0. 0. 0.]
log_prob(vector_good): [0.]
log_prob on 2D matrix of vectors, shape (5, 2) :
[0. 0. 0. 0. 0.]
